In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"D:\IIT\Year 4\IRP\IRP_Project_Folder"
)

DATA_RAW = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

transactions_path = DATA_RAW / "transactions_train.csv"
articles_path = DATA_RAW / "articles.csv"

selected_path = (
    INTERIM_DIR / "stratified_500_articles.csv"
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

print("Weekly demand preprocessing environment ready!")

print("Transactions:", transactions_path.exists())
print("Articles:", articles_path.exists())
print("Selected articles:", selected_path.exists())

Weekly demand preprocessing environment ready!
Transactions: True
Articles: True
Selected articles: True


In [2]:
weekly_demand = con.execute("""
    SELECT
        t.article_id,

        CAST(
            date_trunc('week', CAST(t.t_dat AS DATE))
            AS DATE
        ) AS week_start,

        COUNT(*) AS weekly_demand

    FROM read_csv_auto(?) t

    INNER JOIN read_csv_auto(?) s
        ON t.article_id = s.article_id

    GROUP BY
        t.article_id,
        week_start

    ORDER BY
        t.article_id,
        week_start

""", [
    str(transactions_path),
    str(selected_path)
]).df()

print("Weekly aggregation completed!")

print("Number of article-week records:", len(weekly_demand))

display(weekly_demand.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Weekly aggregation completed!
Number of article-week records: 14046


,article_id,week_start,weekly_demand
0,0189616032,2019-12-09,20
1,0189616032,2019-12-16,45
2,0189616032,2019-12-23,38
3,0189616032,2019-12-30,29
4,0189616032,2020-01-06,32
5,0189616032,2020-01-13,44
6,0189616032,2020-01-20,29
7,0189616032,2020-01-27,20
8,0189616032,2020-02-03,18
9,0189616032,2020-02-10,12


In [3]:
print("Number of unique articles:",
      weekly_demand["article_id"].nunique())

print("Number of unique weeks:",
      weekly_demand["week_start"].nunique())

print("Missing article IDs:",
      weekly_demand["article_id"].isna().sum())

print("Missing weekly demand:",
      weekly_demand["weekly_demand"].isna().sum())

print("Minimum weekly demand:",
      weekly_demand["weekly_demand"].min())

print("Maximum weekly demand:",
      weekly_demand["weekly_demand"].max())

print("Total recorded units:",
      weekly_demand["weekly_demand"].sum())

Number of unique articles: 500
Number of unique weeks: 106
Missing article IDs: 0
Missing weekly demand: 0
Minimum weekly demand: 1
Maximum weekly demand: 1096
Total recorded units: 211157


In [4]:
weekly_complete = con.execute("""
    WITH article_bounds AS (

        SELECT
            article_id,
            MIN(week_start) AS first_week,
            MAX(week_start) AS last_week

        FROM weekly_demand

        GROUP BY article_id
    ),

    article_calendar AS (

        SELECT
            b.article_id,
            CAST(w.week_start AS DATE) AS week_start

        FROM article_bounds b

        CROSS JOIN LATERAL generate_series(
            CAST(b.first_week AS TIMESTAMP),
            CAST(b.last_week AS TIMESTAMP),
            INTERVAL 7 DAY
        ) AS w(week_start)
    )

    SELECT
        c.article_id,
        c.week_start,

        COALESCE(d.weekly_demand, 0) AS weekly_demand

    FROM article_calendar c

    LEFT JOIN weekly_demand d
        ON c.article_id = d.article_id
        AND c.week_start = d.week_start

    ORDER BY
        c.article_id,
        c.week_start

""").df()

print("Completed weekly time series created!")

print("Total records:", len(weekly_complete))

display(weekly_complete.head(15))

Completed weekly time series created!
Total records: 20308


,article_id,week_start,weekly_demand
0,0189616032,2019-12-09,20
1,0189616032,2019-12-16,45
2,0189616032,2019-12-23,38
3,0189616032,2019-12-30,29
4,0189616032,2020-01-06,32
5,0189616032,2020-01-13,44
6,0189616032,2020-01-20,29
7,0189616032,2020-01-27,20
8,0189616032,2020-02-03,18
9,0189616032,2020-02-10,12


In [5]:
article_features = con.execute("""
    SELECT
        article_id,
        product_type_name,
        product_group_name,
        colour_group_name,
        department_name,
        index_name,
        section_name,
        garment_group_name

    FROM read_csv_auto(?)

""", [str(articles_path)]).df()

weekly_dataset = weekly_complete.merge(
    article_features,
    on="article_id",
    how="left",
    validate="many_to_one"
)

print("Article information merged!")

print("Total records:", len(weekly_dataset))

print("Missing product types:",
      weekly_dataset["product_type_name"].isna().sum())

display(weekly_dataset.head(10))

Article information merged!
Total records: 20308
Missing product types: 0


,article_id,week_start,weekly_demand,product_type_name,product_group_name,colour_group_name,department_name,index_name,section_name,garment_group_name
0,0189616032,2019-12-09,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
1,0189616032,2019-12-16,45,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
2,0189616032,2019-12-23,38,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
3,0189616032,2019-12-30,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
4,0189616032,2020-01-06,32,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
5,0189616032,2020-01-13,44,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
6,0189616032,2020-01-20,29,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
7,0189616032,2020-01-27,20,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
8,0189616032,2020-02-03,18,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic
9,0189616032,2020-02-10,12,Leggings/Tights,Garment Lower body,Black,Basic 1,Divided,Divided Basics,Jersey Basic


In [6]:
output_path = (
    INTERIM_DIR / "weekly_demand_500_articles.csv"
)

weekly_dataset.to_csv(output_path, index=False)

print("Weekly demand dataset saved!")
print("Location:", output_path)

Weekly demand dataset saved!
Location: D:\IIT\Year 4\IRP\IRP_Project_Folder\data\interim\weekly_demand_500_articles.csv
